
### What is RAG anyway?


![withoutRAG](https://github.com/user-attachments/assets/649d6101-b63a-4750-997a-b6abc25e5609)

![withRAG](https://github.com/user-attachments/assets/e6dd9c46-0bf9-4c31-bd72-a27939ef82b8)

Retrieval-Augmented Generation (RAG) is a technique primarily used in GenAI applications to improve the quality and accuracy of generated text by LLMs by combining two key processes: retrieval and generation.

### Breaking It Down:
#### Retrieval:

- Before generating a response, the system first looks up relevant information from a large database or knowledge base. This is like searching through a library or the internet to find the most useful facts, articles, or data related to the question or topic.

#### Generation:

- Once the relevant information is retrieved, the system then uses it to help generate a response. This is where the model, like GPT, creates new text (answers, explanations, etc.) based on the retrieved information.

# Install libraries

In [1]:
! pip install python-dotenv langchain langchain-community openai groq tiktoken pinecone-client langchain_pinecone unstructured pdfminer==20191125 pdfminer.six==20221105 pillow_heif unstructured_inference sentence-transformers

In [2]:
from langchain.document_loaders import UnstructuredPDFLoader, OnlinePDFLoader, WebBaseLoader, YoutubeLoader, DirectoryLoader, TextLoader, PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sklearn.metrics.pairwise import cosine_similarity
from langchain_pinecone import PineconeVectorStore
from langchain.embeddings import OpenAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
# from google.colab import userdata
from langchain.schema import Document
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone
from openai import OpenAI
import numpy as np
import tiktoken
import os
from groq import Groq

from dotenv import load_dotenv
import os

load_dotenv()  # take environment variables from .env.

pinecone_api_key = os.environ.get("PINECONE_API_KEY")
openai_api_key = os.environ.get("OPENAI_API_KEY")
groq_api_key = os.environ.get("GROQ_API_KEY")

USER_AGENT environment variable not set, consider setting it to identify your requests.
/Users/vaibhavbhajanka/.pyenv/versions/3.11.9/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Initialize the HuggingFace Embeddings client

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/var/folders/77/hzbmq0jj4t1cjrhd47bsbfnw0000gn/T/ipykernel_71621/3409896792.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [4]:
text = "Hello my name is Vaibhav"

query_result = embeddings.embed_query(text)

In [5]:
query_result

[-0.04870938882231712,
 -0.012672914192080498,
 -0.01893417350947857,
 -0.0020978141110390425,
 -0.086140938103199,
 -0.031322505325078964,
 0.13292951881885529,
 -0.009964508935809135,
 0.047373320907354355,
 -0.009812918491661549,
 -0.016276244074106216,
 -0.12945640087127686,
 0.01986243762075901,
 -0.04093008115887642,
 0.01595877856016159,
 -0.04352167993783951,
 0.07010500878095627,
 0.011876962147653103,
 -0.06188996136188507,
 0.01625143550336361,
 -0.08388068526983261,
 0.04455805569887161,
 -0.04325179383158684,
 -0.0767061859369278,
 -0.07262884080410004,
 -0.0036174680572003126,
 -0.04934884235262871,
 0.059660546481609344,
 -0.03228393569588661,
 -0.054942041635513306,
 0.0571475587785244,
 -0.04725215584039688,
 0.04247008636593819,
 0.0755690485239029,
 0.07180782407522202,
 0.04490362107753754,
 -0.18398970365524292,
 0.023969171568751335,
 -0.05116182565689087,
 0.03697698563337326,
 -0.05382490158081055,
 -0.0619625598192215,
 -0.010159451514482498,
 0.038938086479902

In [6]:
len(query_result)

384

# Initialize the Groq client

In [7]:
# Free Llama 3.1 API via Groq

groq_client = Groq(api_key=groq_api_key)

# Calculating sentence similarity with embeddings

In [8]:
def get_huggingface_embeddings(text, model_name="sentence-transformers/all-MiniLM-L6-v2"):
    model = SentenceTransformer(model_name)
    return model.encode(text)


def cosine_similarity_between_sentences(sentence1, sentence2):
    # Get embeddings for both sentences
    embedding1 = np.array(get_huggingface_embeddings(sentence1))
    embedding2 = np.array(get_huggingface_embeddings(sentence2))

    # Reshape embeddings for cosine_similarity function
    embedding1 = embedding1.reshape(1, -1)
    embedding2 = embedding2.reshape(1, -1)

    print("Embedding for Sentence 1:", embedding1)
    print("\nEmbedding for Sentence 2:", embedding2)

    # Calculate cosine similarity
    similarity = cosine_similarity(embedding1, embedding2)
    return similarity[0][0]


# Example usage
sentence1 = "I like walking to the park"
sentence2 = "I like running to the office"


similarity = cosine_similarity_between_sentences(sentence1, sentence2)
print(f"\n\nCosine similarity between '{sentence1}' and '{sentence2}': {similarity:.4f}")

Embedding for Sentence 1: [[-7.94640393e-04 -4.52190302e-02  5.60034998e-02  4.00062352e-02
   7.82356262e-02 -3.10015166e-03  1.56902835e-01 -1.61644479e-03
   8.40177312e-02  7.29586706e-02 -2.27428079e-02 -1.00336429e-02
  -4.77766320e-02  5.78007028e-02  6.89263046e-02  2.29865871e-03
   3.41052301e-02  8.23903233e-02 -4.47453372e-03  1.18202800e-02
  -7.44136348e-02  2.10828613e-02  1.92200821e-02  5.48400693e-02
  -1.07110791e-01  8.79156962e-02 -1.64800491e-02  6.51676068e-03
  -6.67014392e-05 -4.27564140e-03 -8.20703134e-02  7.05852881e-02
  -1.80556308e-02  3.27348448e-02 -4.36549522e-02  9.93788522e-03
   5.78058138e-02 -6.92315772e-02  4.53141704e-02  4.96660061e-02
  -1.49475625e-02  5.79100400e-02  8.14058036e-02  2.62879208e-03
  -1.49136884e-02 -4.37886119e-02  2.26743054e-02 -3.19028087e-02
   1.00592665e-01  3.10834777e-02  1.30596489e-01  7.27658393e-03
   8.58719833e-03  7.95200281e-03 -7.91899674e-03  4.98277834e-03
  -8.22420791e-02  2.46388819e-02  5.11084683e-02 

# Load in the Data

Learn more about the dataset [here](https://www.kaggle.com/datasets/ayoubcherguelaine/company-documents-dataset)

In [9]:
! kaggle datasets download -d ayoubcherguelaine/company-documents-dataset
! unzip company-documents-dataset.zip

zsh:1: command not found: kaggle
unzip:  cannot find or open company-documents-dataset.zip, company-documents-dataset.zip.zip or company-documents-dataset.zip.ZIP.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
def process_directory(directory_path):
    data = []
    for root, _, files in os.walk(directory_path):
        for file in files:

            file_path = os.path.join(root, file)
            print(f"Processing file: {file_path}")
            loader = PyPDFLoader(file_path)
            data.append({"File": file_path, "Data": loader.load()})

    return data

directory_path = "/content/CompanyDocuments"
documents = process_directory(directory_path)


In [11]:
documents

[]

# Initialize Pinecone

In [12]:
# Make sure to create a Pinecone index with 384 dimensions

index_name = "rag-workshop"

namespace = "company-documents"

vectorstore = PineconeVectorStore(index_name=index_name, embedding=embeddings)

# Insert data into Pinecone

In [ ]:
for document in documents:
  print(document['File'], document['Data'])

In [14]:
document_data = []
for document in documents:
  document_source = document['Data'][0].metadata['source']
  document_content = document['Data'][0].page_content

  file_name = document_source.split('/')[-1]
  folder_names = document_source.split('/')[2:-1]

  print("DOCUMENT SOURCE", document_source)
  print("DOCUMENT CONTENT", document_content)

  doc = Document(
      page_content= f"<Source>\n{document_source}\n</Source>\n\n<Content>\n{document_content}\n<Content>",
      metadata={
          "file_name": file_name,
          "parent_folder": folder_names[-1],
          "folder_names": folder_names
      }
  )

  document_data.append(doc)

In [15]:
document_data

[]

In [16]:
vectorstore_from_texts = PineconeVectorStore.from_documents(
    document_data,
    embeddings,
    index_name=index_name,
    namespace=namespace
)

# Perform RAG

In [18]:
pc = Pinecone(api_key=pinecone_api_key)

pinecone_index = pc.Index(index_name)

In [19]:
query = "What are some common products bought by Mary Saveley"

In [20]:
raw_query_embedding = get_huggingface_embeddings(query)

In [21]:
raw_query_embedding

array([-4.26725410e-02, -6.77617043e-02,  1.35143902e-02, -9.34241898e-03,
       -3.53645012e-02,  1.50140882e-01,  2.53909640e-02, -9.55798384e-03,
       -3.04681454e-02, -8.55065808e-02,  7.04338029e-02, -3.25606088e-03,
        1.76473446e-02, -6.33680969e-02, -1.97863001e-02,  5.99328578e-02,
       -3.36769316e-03,  1.15839489e-01, -2.16769017e-02, -2.27577873e-02,
       -4.50395942e-02,  2.58584339e-02,  4.44792025e-02,  8.40743706e-02,
        2.40686792e-03, -1.55012021e-02,  6.56717177e-03,  2.42127292e-02,
       -4.94251400e-03, -8.99975151e-02, -2.78667081e-02,  1.11753261e-02,
       -3.28647457e-02,  2.51271878e-03,  1.29573736e-02,  3.35337743e-02,
        4.09512483e-02,  3.65742110e-02,  4.78685796e-02, -4.06760816e-03,
       -3.53386886e-02, -1.24937437e-01, -4.32424853e-03, -5.11758914e-03,
       -7.36645162e-02,  2.81967632e-02,  6.48839632e-03,  9.16031674e-02,
        2.23698579e-02, -4.23584208e-02, -6.91725463e-02,  7.57981837e-03,
       -3.12628821e-02, -

In [22]:
top_matches = pinecone_index.query(
    vector=raw_query_embedding.tolist(),
    top_k=10,
    include_metadata=True,
    namespace=namespace
)

In [23]:
top_matches

{'matches': [{'id': 'a948600d-a610-4ab9-8193-d23fb331cf26',
              'metadata': {'file_name': 'purchase_orders_10478.pdf',
                           'folder_names': ['CompanyDocuments',
                                            'PurchaseOrders'],
                           'parent_folder': 'PurchaseOrders',
                           'text': '<Source>\n'
                                   '/content/CompanyDocuments/PurchaseOrders/purchase_orders_10478.pdf\n'
                                   '</Source>\n'
                                   '\n'
                                   '<Content>\n'
                                   'Purchase Orders\n'
                                   'Order ID Order Date Customer Name\n'
                                   '10478 2017-03-18 Mary Saveley\n'
                                   'Products\n'
                                   'Product ID: Product: Quantity: Unit '
                                   'Price:\n'
                         

In [24]:
contexts = [item['metadata']['text'] for item in top_matches['matches']]
#

In [25]:
contexts

['<Source>\n/content/CompanyDocuments/PurchaseOrders/purchase_orders_10478.pdf\n</Source>\n\n<Content>\nPurchase Orders\nOrder ID Order Date Customer Name\n10478 2017-03-18 Mary Saveley\nProducts\nProduct ID: Product: Quantity: Unit Price:\n10 Ikura 20 24.8\nPage 1\n<Content>',
 '<Source>\n/content/CompanyDocuments/PurchaseOrders/purchase_orders_10806.pdf\n</Source>\n\n<Content>\nPurchase Orders\nOrder ID Order Date Customer Name\n10806 2017-12-31 Mary Saveley\nProducts\nProduct ID: Product: Quantity: Unit Price:\n2 Chang 20 19\n65 Louisiana Fiery Hot Pepper Sauce 2 21.05\n74 Longlife Tofu 15 10\nPage 1\n<Content>',
 '<Source>\n/content/CompanyDocuments/PurchaseOrders/purchase_orders_10334.pdf\n</Source>\n\n<Content>\nPurchase Orders\nOrder ID Order Date Customer Name\n10334 2016-10-21 Mary Saveley\nProducts\nProduct ID: Product: Quantity: Unit Price:\n52 Filo Mix 8 5.6\n68 Scottish Longbreads 10 10\nPage 1\n<Content>',
 '<Source>\n/content/CompanyDocuments/PurchaseOrders/purchase_orde

In [26]:
augmented_query = "<CONTEXT>\n" + "\n\n------\n\n".join(contexts[:10])+"\n------\n</CONTEXT>\n\n\n\nMY QUESTION:\n"+query

In [27]:
augmented_query

"<CONTEXT>\n<Source>\n/content/CompanyDocuments/PurchaseOrders/purchase_orders_10478.pdf\n</Source>\n\n<Content>\nPurchase Orders\nOrder ID Order Date Customer Name\n10478 2017-03-18 Mary Saveley\nProducts\nProduct ID: Product: Quantity: Unit Price:\n10 Ikura 20 24.8\nPage 1\n<Content>\n\n------\n\n<Source>\n/content/CompanyDocuments/PurchaseOrders/purchase_orders_10806.pdf\n</Source>\n\n<Content>\nPurchase Orders\nOrder ID Order Date Customer Name\n10806 2017-12-31 Mary Saveley\nProducts\nProduct ID: Product: Quantity: Unit Price:\n2 Chang 20 19\n65 Louisiana Fiery Hot Pepper Sauce 2 21.05\n74 Longlife Tofu 15 10\nPage 1\n<Content>\n\n------\n\n<Source>\n/content/CompanyDocuments/PurchaseOrders/purchase_orders_10334.pdf\n</Source>\n\n<Content>\nPurchase Orders\nOrder ID Order Date Customer Name\n10334 2016-10-21 Mary Saveley\nProducts\nProduct ID: Product: Quantity: Unit Price:\n52 Filo Mix 8 5.6\n68 Scottish Longbreads 10 10\nPage 1\n<Content>\n\n------\n\n<Source>\n/content/CompanyD

In [29]:
system_prompt = f"""You are an expert at understanding and analyzing company data - particularly shipping orders, purchase orders, invoices, and inventory reports.

Answer any questions I have, based on the data provided. Always consider all of the context provided when forming a response.
"""

llm_response = groq_client.chat.completions.create(
    model="llama3-8b-8192",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": augmented_query}
    ]
)

response = llm_response.choices[0].message.content

In [30]:
print(response)

Based on the provided data, I can identify some common products bought by Mary Saveley. After reviewing the purchase orders, I found that Mary Saveley has purchased the following products multiple times:

1. Ikura (Product ID: 10) - purchased in purchase orders 10478 and 10450
2. Louisiana Fiery Hot Pepper Sauce (Product ID: 65) - purchased in purchase orders 10334 and 10251

These products seem to be frequent purchases by Mary Saveley. However, please note that this analysis is limited to the provided data and may not be a comprehensive representation of Mary Saveley's complete purchasing behavior.


# Putting it all together

In [34]:
def perform_rag(query):

  raw_query_embedding = get_huggingface_embeddings(query)
  query_embedding = np.array(raw_query_embedding)
  top_matches = pinecone_index.query(vector=query_embedding.tolist(), top_k=10, include_metadata=True, namespace=namespace)
  # Get the list of retrieved texts
  contexts = [item['metadata']['text'] for item in top_matches['matches']]
  augmented_query = "<CONTEXT>\n" + "\n\n------\n\n".join(contexts[:10])+"\n------\n</CONTEXT>\n\n\n\nMY QUESTION:\n"+query

  # Modify the prompt below as need to improve the response quality
  system_prompt = f"""You are an expert at understanding and analyzing company data - particularly shipping orders, purchase orders, invoices, and inventory reports.

  Answer any questions I have, based on the data provided. Always consider all of the context provided when forming a response.
  """

  res = groq_client.chat.completions.create(
    model="llama3-8b-8192", # llama-3.1-70b-versatile
    messages=[
      {"role": "system", "content": system_prompt},
      {"role": "user", "content": augmented_query}
    ]
  )

  return res.choices[0].message.content

In [35]:
response = perform_rag("What are some trends with Ricardo Adocicados purchase orders?")
print(response)

Based on the provided data, I've analyzed the purchase orders for Ricardo Adocicados. Here are some trends observed:

1. **Frequent order frequency**: Ricardo Adocicados places orders frequently, with at least one order per quarter from 2016 to 2018.

2. **Consistent shipping address**: All orders are shipped to the same address: Av. Copacabana, 267, Rio de Janeiro, South America, 02389-890, Brazil.

3. **Preferred shipper**: The majority of orders (8 out of 9) are shipped by Speedy Express (Shipper ID: 1) or United Package (Shipper ID: 2). Federal Shipping (Shipper ID: 3) only appears once.

4. **Recurring products**: Certain products appear in multiple orders, such as Spegesild, Chang, and Pavlova. However, there is no obvious trend in the ordering pattern for these products.

5. **Order size**: The total price of an order varies significantly, from $95.00 (Order ID 10851) to $1562.5 (Order ID 10877). There doesn't appear to be a consistent pattern in order size.

6. **Employee invol